Welcome to my code for the Kaggle movie review competition.

The logic for this classifier is as follows:

First, identify whether the text is a movie reveiw or not.

Second, identify whether the movie reviews are positive or negative.

Because these two steps will likely benefit from different features, I decided to use two consecutive logicstic regression models.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load dependencies
from typing import Iterator, Iterable, List, Tuple, Text, Union
import numpy as np
import pandas as pd
import re
import html
from scipy.sparse import spmatrix
from sklearn.metrics import accuracy_score, precision_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from langdetect import detect, LangDetectException

# Also note
NDArray = Union[np.ndarray, spmatrix]

In [3]:
np.random.seed(42)

In [4]:
# Read in the training data
df_all = pd.read_csv("train.csv")

## Clean the text

In [5]:
def clean_text(text: str) -> str:
    """
    Cleans raw text by removing HTML tags and.
    """
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
df_all['TEXT'] = df_all['TEXT'].apply(clean_text)

In [7]:
# Add in binary variables for the movie and valence
df_all['MOVIE_LABEL'] = df_all['LABEL'].replace({0: 0, 1: 1, 2: 1})
df_all['VAL_LABEL'] = df_all['LABEL'].replace({0: np.nan, 1: 0, 2: 1})
df_all[0:9]

,ID,TEXT,LABEL,MOVIE_LABEL,VAL_LABEL
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0,0,NaN
1,11848336500074516230,Just getting released from a six month drug re...,1,1,0.0
2,8453485550425672763,I was greatly dissappointed when I popped this...,0,0,NaN
3,13088897910749354342,This is a film that has garnered any interest ...,2,1,1.0
4,4199320520317837800,This is one of the more adorable episodes of t...,1,1,0.0
5,3178394667960127642,"Yes, the 13 year olds(and anyone older)who say...",2,1,1.0
6,4441347800848407828,Superbly researched. Cogent presentation of th...,0,0,NaN
7,3033277043686083675,"St. Teresa is a sad, dying church but the Card...",0,0,NaN
8,12332986805313571562,I don't remember if I finished this book. The ...,0,0,NaN


In [8]:
# See Frequency of labels in all data
print("LABEL: ", df_all['LABEL'].value_counts())
print("MOVIE LABEL: ", df_all['MOVIE_LABEL'].value_counts())
print("VAL LABEL: ", df_all['VAL_LABEL'].value_counts())

LABEL:  0    32289
1    19139
2    18889
Name: LABEL, dtype: int64
MOVIE LABEL:  1    38028
0    32289
Name: MOVIE_LABEL, dtype: int64
VAL LABEL:  0.0    19139
1.0    18889
Name: VAL_LABEL, dtype: int64


In [9]:
# Training file has 70,317 observations.
# Split into training and dev sets (assuming they are randomly ordered)
# dev set = 3%
df_train = df_all[0:68208]
df_dev = df_all[68208:]
print("Training Data Length: ",len(df_train))
print("Dev Data Length: ", len(df_dev))

Training Data Length:  68208
Dev Data Length:  2109


In [10]:
# See Frequency of labels in training vs dev
print("Training\nLABEL:\n", df_train['LABEL'].value_counts())
print("MOVIE LABEL:\n", df_train['MOVIE_LABEL'].value_counts())
print("VAL LABEL:\n", df_train['VAL_LABEL'].value_counts())
print("\n")
print("Dev\nLABEL:\n", df_dev['LABEL'].value_counts())
print("MOVIE LABEL:\n", df_dev['MOVIE_LABEL'].value_counts())
print("VAL LABEL:\n", df_dev['VAL_LABEL'].value_counts())

# All seems equal enough

Training
LABEL:
 0    31343
1    18564
2    18301
Name: LABEL, dtype: int64
MOVIE LABEL:
 1    36865
0    31343
Name: MOVIE_LABEL, dtype: int64
VAL LABEL:
 0.0    18564
1.0    18301
Name: VAL_LABEL, dtype: int64


Dev
LABEL:
 0    946
2    588
1    575
Name: LABEL, dtype: int64
MOVIE LABEL:
 1    1163
0     946
Name: MOVIE_LABEL, dtype: int64
VAL LABEL:
 1.0    588
0.0    575
Name: VAL_LABEL, dtype: int64


As another cleaning step, text not in English will be removed.

Based on a non-systematic review of the texts, most (if not all) non-English text has a label of 0. Therefore, it will be filtered out of the training data, and a simple rule will be used with the classifier to treat non-English texts as 'not a movie review.'

In [11]:
# Function for identifying if text is English. Using langdetect library
def is_english(text: str) -> bool:
    """
    Returns True if the text is English, False otherwise.
    """
    try:
        return detect(text) == 'en'
    except LangDetectException:
        # This catches edge cases where the text is empty, just numbers, or pure punctuation
        return False

The following step may take a bit (3~ minutes for me).

In [12]:
# Add a bool variable to df_train
english = df_train['TEXT'].apply(is_english)

# Filter df_train to keep only the English rows
df_train = df_train[english]
df_train = df_train.reset_index(drop=True)

# Print a quick summary to see how many rows were dropped
print(f"Remaining English rows: {len(df_train)}")

Remaining English rows: 63815


So about 4.4k are removed for not being English...

That's a lot. I feel my assumption that they can all be 0 is not so good, but I'm going to move forward with it for now.

# Identifying movie reviews

This next section will be devoted to creating a classifier that can identify if the given post is a movie review or not.

In [13]:
class TextToFeatures:
    def __init__(self, use_tfidf: bool = False, analyzer: str = 'word', ngram_range: tuple = (1, 1), min_df: int = 1):       
        """
        Initializes an object for converting texts to features.
        
        :param use_tfidf: If True, uses TfidfVectorizer. If False, uses CountVectorizer.
        :param analyzer: used to determine if we're using ngrams ('word') or character-grams ('char_wb')
        :param ngram_range: The lower and upper boundary of the range of n-values for different word n-grams.
        :param min_df: Ignores terms that have a document frequency strictly lower than this threshold.
        """
        if use_tfidf:         # This will be used for the sentiment
            self.vectorizer = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, min_df=min_df)
        else:  # This will be used for the topic
            self.vectorizer = CountVectorizer(analyzer=analyzer, ngram_range=ngram_range)
            
    def fit(self, training_texts: Iterable[Text]) -> None:
        """
        Fits ("trains") a TextToFeature instance on a collection of documents.
        
        The provided training texts are analyzed to determine the vocabulary, 
        i.e., all feature values that the converter will support. 
        Each such feature value will be associated with a unique integer index 
        that may later be accessed via the .index() method.

        It is up to the implementer exactly what features to produce from a
        text, but the features will always include some single words and some
        multi-word expressions (e.g., "need" and "to you").
        
        t2f = TextToFeatures()
        t2f.fit(docs)

        :param training_texts: The training texts.
        """
        self.vectorizer.fit(training_texts)
        
        
    def index(self, feature: Text) -> Union[None, int]:
        """
        Returns the index in the vocabulary of the given feature value.  
        If the features isn't present, return None.

        :param feature: A feature
        :return: The unique integer index associated with the feature or None if not present.
        """
        return self.vectorizer.vocabulary_.get(feature) 
        # Note to self: .get() returns None if the thing is not present

    def transform(self, texts: Iterable[Text]) -> NDArray:
        """
        Creates a feature matrix from a sequence of texts.
        
        
        t2f = TextToFeatures()
        t2f.fit(docs)

        # this produces a NDArray representing our features for the provided doc
        t2f.transform(["Let's meet at Coyote Joe's at 6."])

        Each row of the matrix corresponds to one of the input texts. The value
        at index j of row i is the value in the ith text of the feature
        associated with the unique integer j.

        It is up to the implementer what the value of a feature that is present
        in a text should be, though a common choice is 1. Features that are
        absent from a text will have the value 0.

        :param texts: A sequence of texts.
        :return: A matrix, with one row of feature values for each text.
        """
        return self.vectorizer.transform(texts)

In [14]:
class TextToLabels:
    def __init__(self):
        """
        Initializes an object for converting texts to labels.
        """
        self.encoder = LabelEncoder()

    def fit(self, training_labels: Iterable[Text]) -> None:
        """
        Assigns each distinct label a unique integer.
        
        
        Training labels are analyzed to determine the vocabulary, 
        i.e., all labels that the converter will support. 
        Each such label will be associated with a unique integer index 
        that may later be accessed via the .index() method.

        :param training_labels: The training labels.
        """
        self.encoder.fit(training_labels)
        
    def index(self, label: Text) -> Union[None, int]:
        """Returns the index in the vocabulary of the given label.

        :param label: A label
        :return: The unique integer index associated with the label.
        """
        matches = np.where(self.encoder.classes_ == label)[0] # [0] is bc np.where gives a tuple, and we only want the first part
        # Need to account for matches being empty
        if matches.size == 0:
            return None
        return int(matches[0])

    def transform(self, labels: Iterable[Text]) -> NDArray:
        """
        Creates a label vector from a sequence of labels.

        Each entry in the vector corresponds to one of the input labels. The
        value at index j is the unique integer associated with the jth label.

        :param labels: A sequence of labels.
        :return: A vector, with one entry for each label.
        """
        return self.encoder.transform(labels) 
        
    def __contains__(self, label: Text) -> bool:
        """
        Special "dunder" method to check if a label is known to the TextToLabels instance.
        
        labeler = TextToLabels()
        labeler.fit(["POSITIVE", "NEGATIVE"])

        # should be True:
        "POSITIVE" in labeler 
        
        # should be False:
        "MBOP" in labeler
        
        :return: True if the label was seen in the training data; False otherwise
        """
        return False if self.index(label) is None else True

In [15]:
class Classifier:
    def __init__(self, solver: str = 'lbfgs', max_iter: int = 1000, C: float = 1.0):
        """
        Initalizes a logistic regression classifier.
        """
        self.classifier = LogisticRegression(solver=solver, max_iter=max_iter, C=C)

        
    def train(self, features: NDArray, labels: NDArray) -> None:
        """
        Trains the classifier using the given training examples.

        :param features: A feature matrix, where each row represents a text.
        Such matrices will typically be generated via TextToFeatures.
        :param labels: A label vector, where each entry represents a label.
        Such vectors will typically be generated via TextToLabels.
        """
        self.classifier.fit(features, labels)
    
    # just an alias for "train"
    fit = train
    
    def predict(self, features: NDArray) -> NDArray:
        """Makes predictions for each of the given examples.

        :param features: A feature matrix, where each row represents a text.
        Such matrices will typically be generated via TextToFeatures.
        :return: A prediction vector, where each entry represents a label.
        """
        return self.classifier.predict(features)

## Testing the movie classifier

In [16]:
# Training on movie labels
t2f = TextToFeatures(analyzer='char_wb', ngram_range=(3, 5))
t2l = TextToLabels()
clf = Classifier()

print("Processing & training...")

# Process the text
t2f.fit(df_train['TEXT'])
train_features = t2f.transform(df_train['TEXT'])

# Process the labels
t2l.fit(df_train['MOVIE_LABEL'])
train_labels = t2l.transform(df_train['MOVIE_LABEL'])

# Train the classifier
clf.train(train_features, train_labels)
print("Training complete!")

Processing & training...
Training complete!


Here is a function for using the non-English = 0 (not a movie review) rule.
If it is in Enlgish, it goes to the classifier.

In [17]:
def predict_movie_review(text: str, vectorizer: TextToFeatures, model: Classifier) -> int:
    """
    Predicts if a text is a movie review (1) or not (0), using a language rule first.
    """  
    # RULE: If it's not English (or it's completely empty), it's definitely a 0
    if not text or not is_english(text):
        return 0
        
    # Otherwise, to the classifier!!
    features = vectorizer.transform([text])
    prediction = model.predict(features)
    
    return prediction[0]

In [18]:
# Uses the function above
print("Classifying...")

df_dev['MOVIE_PRED_LABEL'] = df_dev['TEXT'].apply(
    lambda text: predict_movie_review(text, t2f, clf) # lambda function to apply the rule
)

print("Classification complete!")


Classifying...
Classification complete!


/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


In [19]:
# Show a sample of data
print("DataFrame Sample")
print(df_dev[['TEXT', 'MOVIE_LABEL', 'MOVIE_PRED_LABEL']].head(10))
print("\n")

# Calculate metrics
true_labels = df_dev['MOVIE_LABEL']
predictions = df_dev['MOVIE_PRED_LABEL']
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions, zero_division=0)
f1 = f1_score(true_labels, predictions, zero_division=0)

print("Classifier Metrics")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

DataFrame Sample
                                                    TEXT  MOVIE_LABEL  \
68208  Emiliani et al have written a very readable, e...            0   
68209  This movie is full of references. Like "Mad Ma...            1   
68210  Much funnier than expected. I expected bleak, ...            0   
68211  This book hits upon what matters! Education ma...            0   
68212  Hi All, This movie, "The ultimate Gift" is a g...            1   
68213       Great reading very interesting Easy reading.            0   
68214  Not only should you get this book: you should ...            0   
68215  Love the skinnytaste cookbook and was very exc...            0   
68216  If it had been made 2 years later it would hav...            1   
68217  2 WORDS: Academy Award. Nuff said. This film h...            1   

       MOVIE_PRED_LABEL  
68208                 0  
68209                 1  
68210                 0  
68211                 0  
68212                 1  
68213                 0

This seems good enough for now. Moving on!

## The Valence Classifier

Now that I can identify whether a post is about a movie, I will then identify whether it is positive or negative.

To train this classifier, I will filter out any cases that are not about movies. In testing, this will be based on the true labels, but in real use, it will have to be based on the output of the first classifier. 
This is probably a source of error.. but let's hope it's not too great.

In [20]:
# First step is to filter out the rows that aren't about movies.
# ...for the training
df_train_val = df_train[df_train['MOVIE_LABEL'] == 1].copy()
df_train_val = df_train_val.dropna(subset=['TEXT', 'VAL_LABEL']).reset_index(drop=True)

# ...and for the dev set
df_dev_val = df_dev[df_dev['MOVIE_PRED_LABEL'] == 1].copy()
df_dev_val = df_dev_val.dropna(subset=['TEXT', 'VAL_LABEL']).reset_index(drop=True)

print(f"Training on {len(df_train_val)} actual movie reviews.")
print(f"Evaluating on {len(df_dev_val)} predicted movie reviews.")

Training on 36857 actual movie reviews.
Evaluating on 1132 predicted movie reviews.


In [21]:
# Making new instance of the classifier classes
val_t2f = TextToFeatures(use_tfidf=True, analyzer='word', ngram_range=(1, 2), min_df=3)
val_t2l = TextToLabels()
val_clf = Classifier(solver='liblinear')

print("Processing & training...")

# Process the text
val_t2f.fit(df_train_val['TEXT'])
train_val_features = val_t2f.transform(df_train_val['TEXT'])

# Process the labels
val_t2l.fit(df_train_val['VAL_LABEL'])
train_val_labels = val_t2l.transform(df_train_val['VAL_LABEL'])

# Train the classifier
val_clf.train(train_val_features, train_val_labels)
print("Training complete!")

Processing & training...
Training complete!


In [23]:
# Testing the dev set
dev_val_features = val_t2f.transform(df_dev_val['TEXT'])
dev_val_true_labels = val_t2l.transform(df_dev_val['VAL_LABEL'])

# Generate predictions
dev_val_predictions = val_clf.predict(dev_val_features)
df_dev_val['VAL_PRED_LABEL'] = val_t2l.encoder.inverse_transform(dev_val_predictions)

# Calculating metrics
accuracy = accuracy_score(dev_val_true_labels, dev_val_predictions)
precision = precision_score(dev_val_true_labels, dev_val_predictions, average='weighted', zero_division=0)
f1 = f1_score(dev_val_true_labels, dev_val_predictions, average='weighted', zero_division=0)

print("Valence Evaluation Metrics")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

Valence Evaluation Metrics
Accuracy:  0.9108
Precision: 0.9109
F1 Score:  0.9108


## Combining the data to test overall metrics:
This means recoding the valence results to 1 & 2 to compare to the real labels.


In [24]:
# Merge the dev data
df_combined = df_dev.merge(df_dev_val[['ID', 'VAL_PRED_LABEL']], on='ID', how='left')

# Recode the labels to get one equivalent to LABEL
pred_conditions = [
    (df_combined['MOVIE_PRED_LABEL'] == 0) | (df_combined['VAL_PRED_LABEL'].isna()),
    (df_combined['VAL_PRED_LABEL'] == 0),
    (df_combined['VAL_PRED_LABEL'] == 1)
]
choices = [0, 1, 2]
df_combined['FINAL_PRED_LABEL'] = np.select(pred_conditions, choices, default=-1)

# 4. Calculate Final Multi-Class Metrics
# We use average='weighted' to account for class imbalances across the 3 categories
accuracy = accuracy_score(df_combined['LABEL'], df_combined['FINAL_PRED_LABEL'])
precision = precision_score(df_combined['LABEL'], df_combined['FINAL_PRED_LABEL'], average='weighted', zero_division=0)
f1 = f1_score(df_combined['LABEL'], df_combined['FINAL_PRED_LABEL'], average='weighted', zero_division=0)

print("Overall Metrics")
print(f"Overall Accuracy:  {accuracy:.4f}")
print(f"Overall Precision: {precision:.4f}")
print(f"Overall F1 Score:  {f1:.4f}")

Overall Metrics
Overall Accuracy:  0.9374
Overall Precision: 0.9366
Overall F1 Score:  0.9368


This is good enough for me! Now...

# Assessing the test data!

In [25]:
# Read in & clean the test data!
df_test = pd.read_csv("test.csv")
df_test['TEXT'] = df_test['TEXT'].apply(clean_text)

In [26]:
# Starting with the language detection from above
english = df_test['TEXT'].apply(lambda x: is_english(x) if x.strip() else False)


print("Identifying whether it's about a movie...")
# Start with a 0
df_test['MOVIE_PRED_LABEL'] = 0

# Deal with all of English ones using the classifier
if english.sum() > 0:
    # Transform only the valid English text
    test_features = t2f.transform(df_test.loc[english, 'TEXT'])
    # Predict and assign back only to those specific rows
    df_test.loc[english, 'MOVIE_PRED_LABEL'] = clf.predict(test_features)


print("Movie classifcation complete!\nDetermining positive or negative valence...")
# Filter for movie reviews
df_test_val = df_test[df_test['MOVIE_PRED_LABEL'] == 1].copy()


# Transform text using the TF-IDF sentiment vectorizer
test_val_features = val_t2f.transform(df_test_val['TEXT'])
    
# Predict using the sentiment classifier
test_val_predictions = val_clf.predict(test_val_features)
    
# Decode the predictions back to their original format using your label encoder
df_test_val['VAL_PRED_LABEL'] = val_t2l.encoder.inverse_transform(test_val_predictions)



print("Cleaning up predictions...")
# Left merge the sentiment predictions back to the main test dataframe
df_final = df_test[['ID', 'MOVIE_PRED_LABEL']].merge(
    df_test_val[['ID', 'VAL_PRED_LABEL']], 
    on='ID', 
    how='left'
)

# Apply the label logic to return it to the original 0, 1, 2 scale
pred_conditions = [
    (df_final['MOVIE_PRED_LABEL'] == 0) | (df_final['VAL_PRED_LABEL'].isna()),
    (df_final['VAL_PRED_LABEL'] == 0),
    (df_final['VAL_PRED_LABEL'] == 1)
]
choices = [0, 1, 2]
df_final['LABEL'] = np.select(pred_conditions, choices, default=-1)

# Format the final submission dataframe
df_submission = df_final[['ID', 'LABEL']]

print("Submission prepared!")
print(df_submission.head(10))

Identifying whether it's about a movie...
Movie classifcation complete!
Determining positive or negative valence...
Cleaning up predictions...
Submission prepared!
                     ID  LABEL
0   1087873697464833975      1
1   5853461517618045821      1
2   1225516091890726770      2
3  11795874926011119457      0
4  15956464546963841646      0
5  14751864642209654760      0
6   3005217029968835639      0
7  11376939852998541397      2
8   6837960727095302371      0
9   9201125420407700948      2


In [ ]:
# Saving as .csv
df_submission.to_csv("submission.csv", index=False)